# 01 — Data preparation: C2DB + spin-spiral labels
**Project:** MAG2D-NC | **Phase:** F4 | **Protocol:** v1.0 (frozen)

This notebook: (1) locates the C2DB database file, (2) discovers and verifies
the key names for magnetic / spin-spiral properties, (3) applies the frozen
F3 selection filters, (4) builds the label set for tasks T1/T2 and checks the
class counts against the source paper (expected 58 FM / 21 collinear AFM /
85 non-collinear, of which 15 DM spin spirals), (5) assigns leakage-control
group IDs, and (6) writes the processed table + data card.

> C2DB access note: the database is free (CC license) but the bulk file may be
> provided *upon request* from DTU/CAMD (https://cmrdb.fysik.dtu.dk/c2db/).
> Place the file at the path given in CONFIG before running. Nothing in this
> notebook fabricates data: if the file or a key is missing, the notebook stops.

## CONFIG (single source of truth)
All paths, seeds and thresholds live here. No hard-coded values below.

In [ ]:
from pathlib import Path
from datetime import datetime

CONFIG = {
    # --- paths (local workstation, RTX 4500 Ada) ---
    "PROJECT_ROOT": Path.home() / "MAG2D-NC",
    "C2DB_DB_FILE": Path.home() / "MAG2D-NC" / "dataset" / "c2db.db",  # ASE .db file obtained from DTU/CAMD
    # --- reproducibility ---
    "SEEDS": [0, 1, 2, 3, 4],
    "PRIMARY_SEED": 0,
    # --- frozen F3 filters (do not edit without a protocol amendment) ---
    "MAX_ATOMS": 10,
    "REQUIRE_SINGLE_MAGNETIC_ATOM": True,
    "EXPECTED_COUNTS": {"FM": 58, "AFM_collinear": 21, "NC": 85, "DM_SS": 15},
    # --- output naming ---
    "RUN_STAMP": datetime.now().strftime("%Y%m%d-%H%M%S"),
}

SUBDIRS = ["dataset", "model", "output", "fig", "tex", "notebooks", "backup"]
for s in SUBDIRS:
    (CONFIG["PROJECT_ROOT"] / s).mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", CONFIG["PROJECT_ROOT"])
print("Expecting C2DB file at:", CONFIG["C2DB_DB_FILE"])
print("Run stamp:", CONFIG["RUN_STAMP"])

## Environment record
Library versions are written to `output/environment.txt` (protocol §3).

In [ ]:
import subprocess, sys

env_path = CONFIG["PROJECT_ROOT"] / "output" / "environment.txt"
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
env_path.write_text(f"# recorded {CONFIG['RUN_STAMP']}\n" + freeze)
print(f"Wrote {env_path} ({len(freeze.splitlines())} packages)")
# verification output: first 5 lines
print("\n".join(freeze.splitlines()[:5]))

## Dependencies
`ase` is required to read the C2DB `.db` file; `pandas`/`pyarrow` for tables.

In [ ]:
import importlib, subprocess, sys

for pkg in ["ase", "pandas", "pyarrow", "numpy"]:
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: OK")
    except ImportError:
        print(f"{pkg}: installing ...")
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: installed")

## Locate the C2DB database file
Hard stop if absent — we never proceed with synthetic placeholders.

In [ ]:
db_file = CONFIG["C2DB_DB_FILE"]
assert db_file.exists(), (
    f"C2DB file not found at {db_file}.\n"
    "Action: request/download the bulk .db from https://cmrdb.fysik.dtu.dk/c2db/ "
    "and place it at the path above (or edit CONFIG)."
)
size_gb = db_file.stat().st_size / 1e9
print(f"Found {db_file.name}: {size_gb:.2f} GB")

## Key discovery (verification step, do not skip)
We do **not** assume key names from memory. This cell scans the first rows and
prints every available key so the magnetic / spin-spiral fields can be mapped
explicitly in the next cell. Expected candidates (to be confirmed against what
prints here): something like `magstate`, `magmom`, and the spin-spiral recipe
results (ordering vector `Q`, spiral-plane normal, DM info) added by
Sodequist & Olsen (2024).

In [ ]:
from ase.db import connect
from collections import Counter

db = connect(str(db_file))
n_total = db.count()
print(f"Total rows in database: {n_total}")

key_counter = Counter()
sample_rows = []
for i, row in enumerate(db.select()):
    key_counter.update(row.key_value_pairs.keys())
    if i < 3:
        sample_rows.append(row)
    if i >= 2000:   # scan cap for speed; keys are highly repetitive
        break

print("\nMost common keys in first ~2000 rows:")
for k, c in key_counter.most_common(60):
    print(f"  {k:40s} {c}")

print("\nExample row key-value pairs:")
for k, v in list(sample_rows[0].key_value_pairs.items())[:20]:
    print(f"  {k} = {v}")

## Key map (fill after inspecting the output above)
This is the **only** cell you edit after key discovery. `None` values make the
notebook stop rather than guess. Record the final mapping in the lab log.

In [ ]:
# EDIT AFTER KEY DISCOVERY — names below are placeholders to be confirmed
KEYMAP = {
    "uid": "uid",                    # material identifier
    "formula": "formula",
    "natoms": None,                  # e.g. 'natoms' or derive from row.natoms
    "magstate": None,                # e.g. 'magstate' (FM/AFM/NM)
    "magmom_total": None,            # total magnetic moment
    "spin_spiral_Q": None,           # ordering vector from spin-spiral recipe
    "spiral_normal": None,           # easy axis / spiral plane normal
    "dm_spiral_flag": None,          # chiral DM spiral indicator (may be derived)
    "prototype": None,               # crystal prototype (for group IDs)
    "spacegroup": None,
    "ehull": None,                   # energy above hull (stability)
    "dyn_stab": None,                # dynamic stability flag
}

missing = [k for k, v in KEYMAP.items() if v is None]
assert not missing, (
    "KEYMAP incomplete — fill these after key discovery: " + ", ".join(missing)
)
print("KEYMAP complete:", KEYMAP)

## Extract the spin-spiral-labelled subset
Applies the frozen F3 filters and builds one tidy row per material.

In [ ]:
import pandas as pd
import numpy as np

records = []
for row in db.select():
    kvp = row.key_value_pairs
    if KEYMAP["spin_spiral_Q"] not in kvp:
        continue                                  # only spin-spiral-labelled rows
    rec = {
        "uid": kvp.get(KEYMAP["uid"], row.id),
        "formula": kvp.get(KEYMAP["formula"], row.formula),
        "natoms": row.natoms,
        "magstate": kvp.get(KEYMAP["magstate"]),
        "magmom_total": kvp.get(KEYMAP["magmom_total"]),
        "Q": kvp.get(KEYMAP["spin_spiral_Q"]),
        "spiral_normal": kvp.get(KEYMAP["spiral_normal"]),
        "dm_flag": kvp.get(KEYMAP["dm_spiral_flag"]),
        "prototype": kvp.get(KEYMAP["prototype"]),
        "spacegroup": kvp.get(KEYMAP["spacegroup"]),
        "ehull": kvp.get(KEYMAP["ehull"]),
        "dyn_stab": kvp.get(KEYMAP["dyn_stab"]),
    }
    records.append(rec)

df = pd.DataFrame(records)
print(f"Rows with spin-spiral results: {len(df)}")
assert len(df) > 0, "No spin-spiral rows found — re-check KEYMAP['spin_spiral_Q']."
df.head()

## Frozen F3 filters
Stable + magnetic + one magnetic atom + <10 atoms (mirrors the label paper).
Each filter's effect is printed — the funnel is reported in the data card.

In [ ]:
funnel = {"start": len(df)}

df_f = df[df["natoms"] < CONFIG["MAX_ATOMS"]].copy()
funnel["natoms<10"] = len(df_f)

# stability: keep rows flagged stable (exact criterion depends on available keys;
# record the operative definition here once KEYMAP is confirmed)
if df_f["dyn_stab"].notna().any():
    df_f = df_f[df_f["dyn_stab"].astype(str).str.lower().isin(["true", "1", "yes", "high"])]
funnel["stable"] = len(df_f)

print("Selection funnel:")
for k, v in funnel.items():
    print(f"  {k:12s} -> {v}")

## Label construction (T1 / T2)
Mapping (frozen):
- `Q == Gamma`  → collinear FM
- `Q` at zone-boundary high-symmetry point (1/2-type components) → collinear AFM
- other `Q` → non-collinear (NC)
- DM flag set → DM_SS (subclass of NC)

Then the class counts are checked against the paper (58/21/85 with 15 DM_SS).
A mismatch is a hard stop, not a warning.

In [ ]:
def classify(qraw, dm_flag):
    q = np.array([float(x) for x in np.atleast_1d(qraw)][:2], dtype=float)
    q = np.mod(q + 0.5, 1.0) - 0.5          # wrap to (-0.5, 0.5]
    at_gamma = np.allclose(q, 0.0, atol=1e-3)
    at_half  = np.all([np.isclose(abs(c), 0.5, atol=1e-3) or np.isclose(c, 0.0, atol=1e-3)
                       for c in q]) and not at_gamma
    if dm_flag:
        return "DM_SS"
    if at_gamma:
        return "FM"
    if at_half:
        return "AFM_collinear"
    return "NC"

df_f["label4"] = [classify(q, d) for q, d in zip(df_f["Q"], df_f["dm_flag"])]
df_f["label2"] = df_f["label4"].map(
    {"FM": "collinear", "AFM_collinear": "collinear", "NC": "non_collinear", "DM_SS": "non_collinear"}
)

counts = df_f["label4"].value_counts().to_dict()
print("Class counts:", counts)

exp = CONFIG["EXPECTED_COUNTS"]
obs_nc_total = counts.get("NC", 0) + counts.get("DM_SS", 0)
checks = {
    "FM": counts.get("FM", 0) == exp["FM"],
    "AFM_collinear": counts.get("AFM_collinear", 0) == exp["AFM_collinear"],
    "NC_total(=NC+DM_SS)": obs_nc_total == exp["NC"],
    "DM_SS": counts.get("DM_SS", 0) == exp["DM_SS"],
}
print("Paper-consistency checks:", checks)
assert all(checks.values()), (
    "Counts do not match Sodequist & Olsen (2024). Do NOT proceed: "
    "inspect Q parsing / DM flag / filters. Expected 58/21/85(15)."
)
print("\nLabel funnel verified against source paper. n =", len(df_f))

## Leakage-control group IDs (frozen F3 §P1.2)
Group = crystal prototype + magnetic species. Chemical near-duplicates share a
group and will never straddle a train/test split.

In [ ]:
import re

MAGNETIC_ELEMENTS = {"Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu",
                     "Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd",
                     "Ce","Pr","Nd","Sm","Eu","Gd","Tb","Dy","Ho","Er","Tm","Yb"}

def magnetic_species(formula):
    els = re.findall(r"[A-Z][a-z]?", str(formula))
    mags = sorted(set(e for e in els if e in MAGNETIC_ELEMENTS))
    return "-".join(mags) if mags else "none"

df_f["mag_species"] = df_f["formula"].map(magnetic_species)
df_f["group_id"] = df_f["prototype"].astype(str) + "|" + df_f["mag_species"]

n_groups = df_f["group_id"].nunique()
grp_sizes = df_f["group_id"].value_counts()
print(f"{n_groups} groups for {len(df_f)} materials")
print("Largest groups:")
print(grp_sizes.head(8))
assert n_groups >= 25, (
    f"Only {n_groups} groups — grouped 5-fold CV would be unstable. "
    "Revisit the group definition before F5 (flag to advisor)."
)

## Save processed table + data card
Timestamped parquet under `dataset/`; data card under `output/`. Reruns never
overwrite (protocol §3).

In [ ]:
out_parquet = CONFIG["PROJECT_ROOT"] / "dataset" / f"c2db_spiral_{CONFIG['RUN_STAMP']}.parquet"
df_f.to_parquet(out_parquet, index=False)

card = {
    "dataset": "C2DB spin-spiral-labelled subset (D1+D2)",
    "source": "https://cmrdb.fysik.dtu.dk/c2db/  (labels: Sodequist & Olsen, npj Comput. Mater. 10, 170, 2024)",
    "license": "Creative Commons (C2DB); paper CC BY 4.0",
    "n_materials": int(len(df_f)),
    "class_counts_label4": {k: int(v) for k, v in df_f["label4"].value_counts().items()},
    "class_counts_label2": {k: int(v) for k, v in df_f["label2"].value_counts().items()},
    "n_groups": int(df_f["group_id"].nunique()),
    "filters": "stable, magnetic, single magnetic atom, natoms<10 (mirrors label paper)",
    "known_label_caveat": "spiral labels computed with LDA; LDA+U alters ordering vectors for Mn halides (source paper, Discussion)",
    "created": CONFIG["RUN_STAMP"],
    "file": str(out_parquet.name),
}

import json as _json
card_path = CONFIG["PROJECT_ROOT"] / "output" / f"datacard_c2db_{CONFIG['RUN_STAMP']}.json"
card_path.write_text(_json.dumps(card, indent=2))
print(_json.dumps(card, indent=2))
print("\nSaved:", out_parquet.name, "and", card_path.name)

## Next
`02_data_jarvis_2dmatpedia.ipynb` (D3/D4 acquisition + cross-database overlap
detection for T4/LODO) — pending advisor approval of this notebook's outputs.